# Zajęcia 7 — pakowanie modelu, rejestr i karta modelu

Notebook pakuje model razem z sygnaturą, rejestruje jego wersję, wprowadza bramkę jakości opartą na aliasach i generuje kartę modelu z metadanych przebiegu.

Rejestr działa lokalnie, na bazie SQLite w katalogu artefaktów.

Komórki opisane jako **zadanie** wypełniasz samodzielnie. Komórka pod nimi zawiera rozwiązanie ukryte magią `%%skip True`.

## 1. Przygotowanie notebooka

Ta komórka jest gotowa — uruchom ją bez zmian.

In [ ]:
import hashlib
import json
import os
import pickle
import shutil
import subprocess
import sys
from datetime import date
from pathlib import Path

os.environ["MLFLOW_DISABLE_AGENT_HINT"] = "1"

import mlflow
import mlflow.sklearn
import pandas as pd
import skops.io as sio
from IPython.display import display
from mlflow import MlflowClient
from mlflow.models import infer_signature
from sklearn.datasets import load_iris
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "skip_kernel_extension.py").exists():
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

BASE_DIR = REPO_ROOT / "artifacts" / "zajecia_07"
REPORTS_DIR = BASE_DIR / "reports"
ARTIFACT_DIR = BASE_DIR / "mlartifacts"
for directory in (REPORTS_DIR, ARTIFACT_DIR):
    directory.mkdir(parents=True, exist_ok=True)

TRACKING_URI = "sqlite:///" + (BASE_DIR / "mlflow.db").as_posix()
EXPERIMENT_NAME = "zajecia_07_iris"
REGISTERED_NAME = "iris_classifier"
QUALITY_THRESHOLD = 0.90

mlflow.set_tracking_uri(TRACKING_URI)
if mlflow.get_experiment_by_name(EXPERIMENT_NAME) is None:
    mlflow.create_experiment(EXPERIMENT_NAME, artifact_location=ARTIFACT_DIR.as_uri())
mlflow.set_experiment(EXPERIMENT_NAME)
client = MlflowClient()

SEED = 42
FEATURE_COLUMNS = [
    "sepal length (cm)",
    "sepal width (cm)",
    "petal length (cm)",
    "petal width (cm)",
]
TARGET_COLUMN = "target"

DATA_PATH = BASE_DIR / "iris.csv"
load_iris(as_frame=True).frame.to_csv(DATA_PATH, index=False)
data = pd.read_csv(DATA_PATH)
x_train, x_test, y_train, y_test = train_test_split(
    data[FEATURE_COLUMNS],
    data[TARGET_COLUMN],
    test_size=0.3,
    random_state=SEED,
    stratify=data[TARGET_COLUMN],
)

print(f"Rejestr: {TRACKING_URI}")

%load_ext skip_kernel_extension

## 2. Model jako artefakt

Pakujemy model razem z sygnaturą i przykładem wejścia. Sygnaturę wyprowadzamy z rzeczywistych danych funkcją `infer_signature`.

**Zadanie:** napisz funkcję `train_and_log(name, regularization)`, która trenuje model, liczy macro F1 na zbiorze kontrolnym, loguje parametry, metryki, tagi kontekstu oraz model z sygnaturą i przykładem wejścia. Zwróć słownik z `run_id`, `model_uri` i metryką.

In [ ]:
def data_version() -> str:
    digest = hashlib.sha256(DATA_PATH.read_bytes()).hexdigest()
    return digest[:16]


def git_commit() -> str:
    if shutil.which("git") is None:
        return "brak"
    result = subprocess.run(
        ["git", "rev-parse", "--short", "HEAD"], cwd=REPO_ROOT, capture_output=True, text=True
    )
    return result.stdout.strip() if result.returncode == 0 else "brak"


# Zadanie: spakuj i zaloguj model wraz z sygnaturą.


In [ ]:
%%skip True
def train_and_log(name: str, regularization: float) -> dict:
    model = Pipeline(
        steps=[
            ("scaler", StandardScaler()),
            (
                "classifier",
                LogisticRegression(C=regularization, max_iter=500, random_state=SEED),
            ),
        ]
    )
    with mlflow.start_run(run_name=name) as run:
        model.fit(x_train, y_train)
        score = float(f1_score(y_test, model.predict(x_test), average="macro"))

        mlflow.log_params({"model": name, "C": regularization, "seed": SEED})
        mlflow.log_metric("f1_macro", score)
        mlflow.set_tags(
            {
                "data_sha256": data_version(),
                "data_rows": str(len(data)),
                "git_commit": git_commit(),
            }
        )

        info = mlflow.sklearn.log_model(
            sk_model=model,
            name="model",
            signature=infer_signature(x_train, model.predict(x_train)),
            input_example=x_train.head(3),
        )
        return {"run_id": run.info.run_id, "model_uri": info.model_uri, "f1_macro": score}


good = train_and_log("logreg_C1", 1.0)
print(json.dumps(good, indent=2))

## 3. Co naprawdę zawiera artefakt

Zaglądamy do środka. Poza samym modelem artefakt niesie sygnaturę, listę zależności i przykład wejścia — czyli wszystko, czego potrzeba, żeby uruchomić go gdzie indziej. Ta komórka jest gotowa — uruchom ją bez zmian.

In [ ]:
meta = mlflow.models.Model.load(good["model_uri"])
print("SYGNATURA")
print(meta.signature)

local_copy = Path(mlflow.artifacts.download_artifacts(good["model_uri"]))
print("\nZAWARTOŚĆ ARTEFAKTU")
for item in sorted(local_copy.rglob("*")):
    if item.is_file():
        print(f"  {item.relative_to(local_copy)}  ({item.stat().st_size} B)")

## 4. Punkt kontrolny 1 — egzekwowanie sygnatury

Podajemy modelowi dane z brakującą kolumną oraz z błędnym typem.

**Do opisania:** w którym momencie pojawił się błąd i co stałoby się bez sygnatury?

In [ ]:
served = mlflow.pyfunc.load_model(good["model_uri"])
print(f"Poprawne wejście: {served.predict(x_test.head(3))}\n")

missing_column = x_test.head(3).drop(columns=["petal width (cm)"])
try:
    served.predict(missing_column)
except Exception as error:
    print(f"Brakująca kolumna -> {type(error).__name__}")
    print(f"  {str(error).splitlines()[0][:120]}\n")

wrong_type = x_test.head(3).copy()
wrong_type["petal width (cm)"] = wrong_type["petal width (cm)"].astype(str)
try:
    served.predict(wrong_type)
except Exception as error:
    print(f"Błędny typ kolumny -> {type(error).__name__}")
    print(f"  {str(error).splitlines()[0][:120]}")

## 5. Ryzyko formatów serializacji

Wczytanie pliku `pickle` oznacza **wykonanie zapisanego w nim kodu**. Poniżej nieszkodliwa demonstracja tego mechanizmu oraz bezpieczniejsza alternatywa dla modeli scikit-learn.

**Do opisania:** dlaczego skanowanie zależności nie wykryje złośliwego artefaktu modelu?

In [ ]:
class NieszkodliwyLadunek:
    def __reduce__(self):
        return (print, ("  <-- ten kod wykonał się w trakcie wczytywania artefaktu",))


print("Wczytywanie pliku pickle:")
pickle.loads(pickle.dumps(NieszkodliwyLadunek()))

safe_path = BASE_DIR / "model.skops"
model = mlflow.sklearn.load_model(good["model_uri"])
sio.dump(model, safe_path)

untrusted = sio.get_untrusted_types(file=safe_path)
print(f"\nFormat skops — typy wymagające zatwierdzenia: {untrusted}")
restored = sio.load(safe_path, trusted=untrusted)
print(f"Model wczytany bezpiecznie, predykcja: {restored.predict(x_test.head(3))}")

## 6. Rejestr modeli

Rejestr oddziela „co próbowaliśmy” od „co jest używane”. Wersja jest niezmienna, alias jest ruchomym wskaźnikiem.

**Zadanie:** napisz funkcję `register(model_uri, run_id, score)`, która rejestruje nową wersję pod nazwą `REGISTERED_NAME`, nadaje jej alias `candidate` oraz zapisuje tagi z metryką i wersją danych. Zwróć numer wersji.

In [ ]:
# Zadanie: zarejestruj wersję modelu i nadaj jej alias candidate.
# Wskazówka: mlflow.register_model, client.set_registered_model_alias,
# client.set_model_version_tag, client.update_model_version.


In [ ]:
%%skip True
def register(model_uri: str, run_id: str, score: float) -> int:
    version = mlflow.register_model(model_uri, REGISTERED_NAME)
    client.set_model_version_tag(REGISTERED_NAME, version.version, "f1_macro", f"{score:.4f}")
    client.set_model_version_tag(REGISTERED_NAME, version.version, "data_sha256", data_version())
    client.set_model_version_tag(REGISTERED_NAME, version.version, "git_commit", git_commit())
    client.update_model_version(
        REGISTERED_NAME,
        version.version,
        description=f"Model z przebiegu {run_id}, macro F1 = {score:.4f}",
    )
    client.set_registered_model_alias(REGISTERED_NAME, "candidate", version.version)
    return int(version.version)


good_version = register(good["model_uri"], good["run_id"], good["f1_macro"])
print(f"Zarejestrowano wersję {good_version} z aliasem 'candidate'.")

## 7. Bramka jakości

Bramka ma być kodem, a nie ustaleniem. Reguły:

1. metryka nie niższa niż próg bezwzględny (`QUALITY_THRESHOLD`);
2. metryka nie gorsza od obecnego `champion`;
3. komplet wymaganych metadanych: `f1_macro`, `data_sha256`, `git_commit`.

**Zadanie:** zaimplementuj `promote(version)`, która sprawdza reguły, zgłasza `ValueError` z czytelnym powodem przy niepowodzeniu, a przy powodzeniu nadaje alias `champion` (albo `challenger`, gdy champion już istnieje i jest lepszy).

In [ ]:
# Zadanie: zaimplementuj bramkę jakości.


In [ ]:
%%skip True
REQUIRED_TAGS = ("f1_macro", "data_sha256", "git_commit")


def current_champion():
    try:
        return client.get_model_version_by_alias(REGISTERED_NAME, "champion")
    except Exception:
        return None


def promote(version: int) -> str:
    info = client.get_model_version(REGISTERED_NAME, str(version))

    missing = [tag for tag in REQUIRED_TAGS if tag not in info.tags]
    if missing:
        raise ValueError(f"Wersja {version}: brak wymaganych metadanych {missing}.")

    score = float(info.tags["f1_macro"])
    if score < QUALITY_THRESHOLD:
        raise ValueError(
            f"Wersja {version}: macro F1 = {score:.4f} poniżej progu {QUALITY_THRESHOLD}."
        )

    champion = current_champion()
    if champion is None:
        client.set_registered_model_alias(REGISTERED_NAME, "champion", str(version))
        return "champion"

    champion_score = float(champion.tags["f1_macro"])
    if score < champion_score:
        raise ValueError(
            f"Wersja {version}: macro F1 = {score:.4f} gorsze niż champion "
            f"(wersja {champion.version}, {champion_score:.4f})."
        )

    client.set_registered_model_alias(REGISTERED_NAME, "challenger", str(version))
    return "challenger"


print(f"Wersja {good_version} otrzymała alias: {promote(good_version)}")

## 8. Punkt kontrolny 2 — model poniżej progu

Trenujemy model nadmiernie regularyzowany, rejestrujemy go i próbujemy promować.

**Do opisania:** która reguła bramki zadziałała i co by się stało, gdyby bramka istniała wyłącznie w dokumentacji?

In [ ]:
weak = train_and_log("logreg_C1e-6", 1e-6)
weak_version = register(weak["model_uri"], weak["run_id"], weak["f1_macro"])
print(f"Zarejestrowano wersję {weak_version}, macro F1 = {weak['f1_macro']:.4f}\n")

try:
    promote(weak_version)
except ValueError as error:
    print(f"Promocja zablokowana: {error}")

print(f"\nChampion pozostał wersją: {current_champion().version}")

## 9. Punkt kontrolny 3 — przeniesienie aliasu

Konsument ładuje model **po aliasie**, nigdy po numerze wersji. Sprawdzamy, ile trzeba zmienić po jego stronie, gdy zmienia się wersja produkcyjna.

**Do opisania:** ile zmian wymagał kod konsumenta i jak wyglądałoby wycofanie zmiany?

In [ ]:
CONSUMER_URI = f"models:/{REGISTERED_NAME}@champion"


def consumer_prediction() -> dict:
    version = client.get_model_version_by_alias(REGISTERED_NAME, "champion").version
    model = mlflow.pyfunc.load_model(CONSUMER_URI)
    return {"wersja": version, "predykcje": list(model.predict(x_test.head(5)))}


before = consumer_prediction()
print(f"Przed: {before}")

newer = train_and_log("logreg_C10", 10.0)
newer_version = register(newer["model_uri"], newer["run_id"], newer["f1_macro"])
client.set_registered_model_alias(REGISTERED_NAME, "champion", str(newer_version))

after = consumer_prediction()
print(f"Po:    {after}")
print(f"\nKod konsumenta ({CONSUMER_URI}) nie zmienił się ani o znak.")

client.set_registered_model_alias(REGISTERED_NAME, "champion", str(good_version))
print(f"Wycofanie: champion ponownie wskazuje wersję {current_champion().version}.")

## 10. Karta modelu

Karta modelu powstaje **z metadanych**, a nie z pamięci autora. Dzięki temu nie rozjedzie się z rzeczywistością przy kolejnej wersji.

**Zadanie:** wygeneruj `reports/model_card.md` zawierający przeznaczenie, dane, wyniki, ograniczenia, zastosowania niezalecane oraz sekcję identyfikowalności z wersją, przebiegiem, commitem i sumą kontrolną danych.

In [ ]:
# Zadanie: wygeneruj kartę modelu z metadanych rejestru i przebiegu.


In [ ]:
%%skip True
def build_model_card() -> str:
    version = client.get_model_version_by_alias(REGISTERED_NAME, "champion")
    run = mlflow.get_run(version.run_id)

    return f"""# Karta modelu: {REGISTERED_NAME}

## Przeznaczenie

Klasyfikacja gatunku irysa na podstawie czterech pomiarów morfologicznych.
Odbiorca: przykładowa aplikacja dydaktyczna. Model nie jest przeznaczony do użycia produkcyjnego.

## Dane treningowe

- Zbiór: iris, {run.data.tags.get('data_rows')} wierszy
- Suma kontrolna: `{run.data.tags.get('data_sha256')}`
- Cechy: {', '.join(FEATURE_COLUMNS)}

## Wyniki

- macro F1 (zbiór kontrolny): {float(version.tags['f1_macro']):.4f}
- Próg akceptacji: {QUALITY_THRESHOLD}
- Parametry: {json.dumps(run.data.params, ensure_ascii=False)}

## Ograniczenia

- Model zakłada dokładnie cztery cechy liczbowe w zakresach ze zbioru treningowego.
- Zbiór jest mały i idealnie zbalansowany; wynik nie przenosi się na dane rzeczywiste.
- Model nie rozpoznaje obserwacji spoza dziedziny — zwróci klasę także dla bezsensownego wejścia.

## Zastosowania niezalecane

- Jakiekolwiek decyzje dotyczące osób.
- Dane z innego źródła niż opisane powyżej, bez ponownej walidacji kontraktu.

## Identyfikowalność

| Element | Wartość |
|---|---|
| Alias | `champion` |
| Wersja | {version.version} |
| Przebieg | `{version.run_id}` |
| Commit | `{run.data.tags.get('git_commit')}` |
| Wersja danych | `{run.data.tags.get('data_sha256')}` |
| Data karty | {date.today().isoformat()} |
"""


card = build_model_card()
(REPORTS_DIR / "model_card.md").write_text(card, encoding="utf-8")
print(card)

## 11. Łańcuch identyfikowalności

Zamykamy łańcuch: od aliasu, przez wersję i przebieg, aż do commita i sumy kontrolnej danych. Ta komórka jest gotowa — uruchom ją bez zmian.

In [ ]:
version = client.get_model_version_by_alias(REGISTERED_NAME, "champion")
run = mlflow.get_run(version.run_id)

lineage = {
    "alias": "champion",
    "registered_model": REGISTERED_NAME,
    "version": version.version,
    "run_id": version.run_id,
    "params": run.data.params,
    "metrics": {key: round(value, 4) for key, value in run.data.metrics.items()},
    "git_commit": run.data.tags.get("git_commit"),
    "data_sha256": run.data.tags.get("data_sha256"),
    "model_uri": version.source,
}

(REPORTS_DIR / "lineage.json").write_text(
    json.dumps(lineage, indent=2, ensure_ascii=False), encoding="utf-8"
)
print(json.dumps(lineage, indent=2, ensure_ascii=False))

print("\nWszystkie wersje w rejestrze:")
display(
    pd.DataFrame(
        [
            {
                "wersja": item.version,
                "f1_macro": item.tags.get("f1_macro"),
                "aliasy": ", ".join(item.aliases) or "-",
            }
            for item in client.search_model_versions(f"name='{REGISTERED_NAME}'")
        ]
    ).sort_values("wersja")
)

## 12. Podsumowanie

Model przestał być plikiem. Ma sygnaturę, wersję, alias, kartę i pełny łańcuch pochodzenia. Bramka jakości jest kodem, więc nie da się jej ominąć w pośpiechu.

Czego nadal brakuje:

- model jest dostępny tylko z poziomu Pythona, nie z innej aplikacji (zajęcia 8);
- nikt nie sprawdza automatycznie, czy nowa wersja przeszła testy przed rejestracją (zajęcia 9);
- nie ma procedury bezpiecznej zmiany wersji na produkcji ani wycofania (zajęcia 10).

### Zadanie do własnego projektu

1. Artefakt modelu z sygnaturą i przykładem wejścia.
2. Zarejestrowana wersja pod stabilną nazwą.
3. Aliasy `champion` i `challenger`; konsument ładuje model wyłącznie po aliasie.
4. Bramka jakości jako funkcja zgłaszająca wyjątek.
5. Karta modelu generowana z metadanych.
6. Zapis lineage: alias → wersja → przebieg → commit → suma kontrolna danych.
7. Opis używanego formatu serializacji wraz z oceną ryzyka.